#Lab 10

1 RNN

In [1]:
import torch
import torch.nn as nn
import numpy as np

# Dataset
text = """artificial intelligence systems learn patterns from data.
sequence models process information step by step.
recurrent neural networks are useful for sequence prediction.
lstm networks handle long term dependencies.
deep learning models improve sequence learning.
generative models create new samples from learned patterns.
language models predict the next word in a sentence.
sequence generation is used in chatbots and assistants.
machine learning helps computers learn automatically.
training data improves model accuracy.
neural networks simulate human brain structures.
optimization algorithms improve learning efficiency.
technology is transforming modern education.
online learning platforms use artificial intelligence.
students benefit from intelligent tutoring systems.
automation improves productivity and decision making."""

# Character-level processing
chars = sorted(list(set(text)))
char_to_ix = {ch:i for i,ch in enumerate(chars)}
ix_to_char = {i:ch for ch,i in char_to_ix.items()}

# Encode text
data = [char_to_ix[ch] for ch in text]

# Prepare sequences
seq_length = 40
X, Y = [], []

for i in range(len(data) - seq_length):
    X.append(data[i:i+seq_length])
    Y.append(data[i+1:i+seq_length+1])

X = torch.tensor(X)
Y = torch.tensor(Y)

# RNN Model
class RNNModel(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.rnn = nn.RNN(vocab_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = self.fc(out)
        return out

# One-hot encoding
def one_hot(x, vocab_size):
    return torch.nn.functional.one_hot(x, num_classes=vocab_size).float()

model = RNNModel(len(chars), 128)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training
for epoch in range(50):
    inputs = one_hot(X, len(chars))
    outputs = model(inputs)
    loss = criterion(outputs.view(-1, len(chars)), Y.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")

# Generate text
def generate(seed, length=100):
    model.eval()
    input_seq = torch.tensor([char_to_ix[c] for c in seed]).unsqueeze(0)

    for _ in range(length):
        inp = one_hot(input_seq[:, -seq_length:], len(chars))
        out = model(inp)
        probs = torch.softmax(out[0, -1], dim=0)
        char_ix = torch.multinomial(probs, 1).item()

        input_seq = torch.cat([input_seq, torch.tensor([[char_ix]])], dim=1)

    return ''.join(ix_to_char[i.item()] for i in input_seq[0])

print(generate("machine learning "))

Epoch 0, Loss: 3.340933322906494
Epoch 10, Loss: 2.731764554977417
Epoch 20, Loss: 2.023061752319336
Epoch 30, Loss: 1.536043405532837
Epoch 40, Loss: 1.0591049194335938
machine learning nyxt prwven ieproteslencelcicats models crfrhm letproves msterats.
onep.aleor ingro timproc.
lmad in


2 LSTM

In [2]:
import torch
import torch.nn as nn

# Same dataset and preprocessing as above

class LSTMModel(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.lstm = nn.LSTM(vocab_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out)
        return out

model = LSTMModel(len(chars), 128)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

# Training
for epoch in range(50):
    inputs = one_hot(X, len(chars))
    outputs = model(inputs)
    loss = criterion(outputs.view(-1, len(chars)), Y.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")

# Generate
print(generate("artificial intelligence "))

Epoch 0, Loss: 3.3427071571350098
Epoch 10, Loss: 2.957068920135498
Epoch 20, Loss: 2.9046032428741455
Epoch 30, Loss: 2.795074224472046
Epoch 40, Loss: 2.5517258644104004
artificial intelligence ormimfourr scrorn  arrk n.tse nsis.uapat.imrspnedin itimtins 
rimitniqlimrps ngetmsell peetoprgile.f


3 Transformer

In [3]:
import torch
import torch.nn as nn
import math

# Word-level tokenization
words = text.split()
vocab = list(set(words))
word_to_ix = {w:i for i,w in enumerate(vocab)}
ix_to_word = {i:w for w,i in word_to_ix.items()}

data = [word_to_ix[w] for w in words]

seq_length = 5
X, Y = [], []

for i in range(len(data) - seq_length):
    X.append(data[i:i+seq_length])
    Y.append(data[i+1:i+seq_length+1])

X = torch.tensor(X)
Y = torch.tensor(Y)

# Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)

        for pos in range(max_len):
            for i in range(0, d_model, 2):
                pe[pos, i] = math.sin(pos / (10000 ** ((2*i)/d_model)))
                if i+1 < d_model:
                    pe[pos, i+1] = math.cos(pos / (10000 ** ((2*(i+1))/d_model)))

        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

# Transformer Model
class TransformerModel(nn.Module):
    def __init__(self, vocab_size, d_model=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=4)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        x = self.pos(x)
        x = x.permute(1, 0, 2)
        x = self.transformer(x)
        x = x.permute(1, 0, 2)
        return self.fc(x)

model = TransformerModel(len(vocab))
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

# Training
for epoch in range(100):
    outputs = model(X)
    loss = criterion(outputs.view(-1, len(vocab)), Y.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")

# Generate sequence
def generate_words(seed, length=10):
    model.eval()
    input_seq = [word_to_ix[w] for w in seed.split()]

    for _ in range(length):
        inp = torch.tensor(input_seq[-seq_length:]).unsqueeze(0)
        out = model(inp)
        probs = torch.softmax(out[0, -1], dim=0)
        idx = torch.multinomial(probs, 1).item()
        input_seq.append(idx)

    return ' '.join(ix_to_word[i] for i in input_seq)

print(generate_words("machine learning"))


/tmp/ipykernel_1087/1779800344.py:48: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)


Epoch 0, Loss: 4.632932186126709
Epoch 20, Loss: 0.15768007934093475
Epoch 40, Loss: 0.01565687730908394
Epoch 60, Loss: 0.0066733649000525475
Epoch 80, Loss: 0.004735779017210007
machine learning helps computers learn automatically. training data improves model accuracy. neural
